In [ ]:
import pymc as pm
import arviz as az
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# --- 1. Load the Data (The "Integration" Proof) ---
print("Loading processed datasets...")

ROOT = Path.cwd()  # VS Code notebooks usually run from workspace root
DATA = ROOT / "data" / "processed"

df_health = pd.read_csv(DATA / "gbd_clean_nordic.csv")
df_weather = pd.read_csv(DATA / "norway_weather_clean.csv")

# --- 2. Merge the Data ---
df_merged = pd.merge(df_health, df_weather, on="year")

print(f"Merged Dataset Shape: {df_merged.shape}")
print(df_merged.head())

# --- Safety checks for expected columns ---
# Adjust these names if your processed files use different column names
if "deaths_per_100k" not in df_merged.columns and "val" in df_merged.columns:
    df_merged = df_merged.rename(columns={"val": "deaths_per_100k"})

required = {"country", "year", "temperature", "deaths_per_100k"}
missing = required - set(df_merged.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}. Found columns: {list(df_merged.columns)}")

df_merged = df_merged.dropna(subset=["temperature", "deaths_per_100k"]).copy()

coords = {
    "country": df_merged["country"].unique(),
    "obs_id": np.arange(len(df_merged)),
}

with pm.Model(coords=coords) as hierarchical_model:
    temp_idx = pm.Data("temp_idx", df_merged["temperature"].astype(float).values)
    country_idx = pd.Categorical(df_merged["country"], categories=coords["country"]).codes

    alpha = pm.Normal("alpha", mu=0, sigma=1)

    mu_beta = pm.Normal("mu_beta", mu=0, sigma=1)
    sigma_beta = pm.HalfNormal("sigma_beta", sigma=1)
    beta = pm.Normal("beta", mu=mu_beta, sigma=sigma_beta, dims="country")

    mu = pm.math.exp(alpha + beta[country_idx] * temp_idx)

    sigma_obs = pm.HalfNormal("sigma_obs", sigma=1)
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma_obs, observed=df_merged["deaths_per_100k"].astype(float).values)

    print("Sampling from the posterior... (MCMC)")
    idata = pm.sample(1000, tune=1000, target_accept=0.9, random_seed=42, return_inferencedata=True)

plt.figure(figsize=(10, 6))
az.plot_forest(idata, var_names=["beta"], combined=True)
plt.title("Estimated Effect of Temperature on Cardiovascular Mortality (by Country)")
plt.xlabel("Effect Size (Beta Coefficient)")
plt.axvline(x=0, color="red", linestyle="--")
plt.show()

print("Interpretation:")
print("If the distribution is to the right of the red line, higher temperatures increase mortality.")
print("This hierarchical structure allows us to estimate effects even for countries with noisy data.")
